# ♻️ WasteVision Studio — 03: CNN Architecture & Training Pipeline
### بناء الشبكة العصبية الالتفافية وتدريبها بنسب متوازنة
يتضمن تقسيم البيانات طبقياً، حساب أوزان الفئات، محاكاة أحجام الموترات ثلاثية الأبعاد، وتدريب النموذج مع EarlyStopping و ReduceLROnPlateau.


In [ ]:
import sys
from pathlib import Path
import numpy as np
import matplotlib.pyplot as plt
import tensorflow as tf
import plotly.graph_objects as go
from sklearn.utils.class_weight import compute_class_weight

sys.path.append("..")
from core.dataset_service import DatasetService
from core.cnn_builder import CNNBuilder


In [ ]:
# 1. التقسيم الطبقي المتوازن وحساب أوزان الفئات
DATA_DIR = Path("../data/dataset/dataset-resized")
if not DATA_DIR.exists(): DATA_DIR = Path("../data/dataset")

details = DatasetService().scan(DATA_DIR)
class_names = details["classes"]
train_rec, val_rec, test_rec = DatasetService.stratified_split(details["records"], test_percent=15, validation_percent=15)

print(f"تقسيم البيانات -> تدريب: {len(train_rec)} | تحقق: {len(val_rec)} | اختبار: {len(test_rec)}")

class_weights = CNNBuilder.compute_balanced_weights(class_names, train_rec)
print("أوزان الفئات التعويضية:", {class_names[k]: round(v, 2) for k, v in class_weights.items()})


In [ ]:
# 2. محاكاة ثلاثية الأبعاد لأحجام الموترات عبر طبقات CNN (Conceptual 3D Tensors)
stages = [(160,160,3,"Input"),(160,160,32,"Conv 32"),(80,80,32,"Pool"),(80,80,64,"Conv 64"),(40,40,64,"Pool"),(40,40,128,"Conv 128"),(20,20,128,"Pool"),(1,1,6,"Classes")]
fig = go.Figure()
for i, (h, w, ch, name) in enumerate(stages):
    dx, dy, dz = max(w/160, 0.08), max(h/160, 0.08), max(ch/128, 0.05)
    x0 = i * 1.45
    verts = [(x0,0,0),(x0+dx,0,0),(x0+dx,dy,0),(x0,dy,0),(x0,0,dz),(x0+dx,0,dz),(x0+dx,dy,dz),(x0,dy,dz)]
    faces = [(0,1,2),(0,2,3),(4,5,6),(4,6,7),(0,1,5),(0,5,4),(1,2,6),(1,6,5),(2,3,7),(2,7,6),(3,0,4),(3,4,7)]
    xx, yy, zz = zip(*verts); ii, jj, kk = zip(*faces)
    fig.add_trace(go.Mesh3d(x=xx, y=yy, z=zz, i=ii, j=jj, k=kk, opacity=0.82, color=["#4cc9f0","#4895ef","#4361ee","#3f37c9","#7209b7","#b5179e","#f72585","#ffb703"][i], name=f"{name}: {h}x{w}x{ch}"))
fig.update_layout(title="CNN Tensor Flow — Conceptual 3D Architecture", height=520)
fig.show()


In [ ]:
# 3. بناء وتدريب النموذج
model = CNNBuilder.build(tf, class_count=len(class_names), image_size=160, filters=32, dropout=0.35, learning_rate=0.001, augmentation=True)
model.summary()


In [ ]:
# 4. تجهيز tf.data ورسم منحنيات التعلم (Loss & Accuracy)
from core.trainer_service import TrainingManager
tm = TrainingManager(DatasetService())
train_ds = tm.build_tf_dataset(tf, train_rec, class_names, 160, 16, training=True)
val_ds = tm.build_tf_dataset(tf, val_rec, class_names, 160, 16)

callbacks = [
    tf.keras.callbacks.EarlyStopping(monitor="val_loss", patience=3, restore_best_weights=True),
    tf.keras.callbacks.ReduceLROnPlateau(monitor="val_loss", factor=0.5, patience=2)
]

history = model.fit(train_ds, validation_data=val_ds, epochs=12, class_weight=class_weights, callbacks=callbacks)

fig, axes = plt.subplots(1, 2, figsize=(11, 4))
axes[0].plot(history.history["loss"], label="Train Loss", color="#0c9b79")
axes[0].plot(history.history["val_loss"], label="Val Loss", color="#d18a19")
axes[0].set_title("Loss Curves"); axes[0].legend(); axes[0].grid(alpha=0.3)

axes[1].plot(history.history["accuracy"], label="Train Acc", color="#1688c7")
axes[1].plot(history.history["val_accuracy"], label="Val Acc", color="#d18a19")
axes[1].set_title("Accuracy Curves"); axes[1].legend(); axes[1].grid(alpha=0.3)
plt.tight_layout()
plt.show()
